In [ ]:
# ============================================================
# DENGUE HOST–VIRUS DYNAMICS PROJECT
# NOTEBOOK 01 — DATA INGESTION AND AUDIT
# ============================================================
#
# Working project title:
#
# Temporal Decoupling of Viral Clearance and Host Recovery
# Defines Distinct Clinical States in Dengue
#
# Primary scientific question:
#
# Does temporal decoupling between dengue viral burden and
# host physiological recovery identify clinically meaningful
# disease states and predict subsequent deterioration?
#
# Study design:
# Longitudinal observational analysis of human dengue infection
#
# Main biological variables:
#   Viral burden     -> viremia
#   Host perturbation -> platelet dynamics
#
# Clinical outcomes:
#   Severe dengue
#   Plasma leakage
#
# IMPORTANT:
# Notebook 01 contains NO predictive modelling.
# Its purpose is data integrity, cohort definition,
# temporal structure and missingness assessment.
# ============================================================

PROJECT_NAME = "Dengue Host–Virus Dynamics"
NOTEBOOK = "01_Data_Audit"
RANDOM_SEED = 10010

print(PROJECT_NAME)
print("Notebook:", NOTEBOOK)
print("Random seed:", RANDOM_SEED)

In [ ]:
import os
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

np.random.seed(RANDOM_SEED)

print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("System :", platform.system(), platform.release())
print("Current working directory:")
print(Path.cwd())

In [ ]:
DATA_FILENAME = "Longitudinal viremia and platelet 22May2024.csv"

candidate_locations = [
    Path.cwd() / DATA_FILENAME,
    Path.cwd().parent / DATA_FILENAME,
    Path.cwd() / "data" / DATA_FILENAME,
    Path.cwd().parent / "data" / DATA_FILENAME,
    Path.cwd().parent.parent / "data" / DATA_FILENAME,
]

DATA_PATH = None

for path in candidate_locations:
    if path.exists():
        DATA_PATH = path.resolve()
        break

if DATA_PATH is None:
    print("Dataset not found automatically.")
    print("\nChecked:")
    for p in candidate_locations:
        print(" -", p)
else:
    print("Dataset found:")
    print(DATA_PATH)

In [ ]:
assert DATA_PATH is not None, "Dataset path has not been resolved."

df_raw = pd.read_csv(DATA_PATH)

print("=" * 70)
print("RAW DATASET")
print("=" * 70)

print(f"Rows    : {df_raw.shape[0]:,}")
print(f"Columns : {df_raw.shape[1]:,}")

display(df_raw.head())

In [ ]:
print("COLUMN NAMES")
print("=" * 70)

for i, col in enumerate(df_raw.columns, start=1):
    print(f"{i:02d}. {col}")

In [ ]:
EXPECTED_COLUMNS = [
    "Study",
    "Code",
    "Age",
    "Sex",
    "Enrol_DOI",
    "Serotype",
    "Serology",
    "DOI",
    "PLT",
    "vir",
    "Severe_dengue",
    "doi_severe",
    "Leakage",
    "doi_leakage",
]

actual = list(df_raw.columns)

missing_expected = [c for c in EXPECTED_COLUMNS if c not in actual]
unexpected = [c for c in actual if c not in EXPECTED_COLUMNS]

print("Missing expected columns:")
print(missing_expected if missing_expected else "None")

print("\nUnexpected columns:")
print(unexpected if unexpected else "None")

print("\nExact expected structure:",
      actual == EXPECTED_COLUMNS)

In [ ]:
audit_types = pd.DataFrame({
    "Column": df_raw.columns,
    "Data_Type": df_raw.dtypes.astype(str).values,
    "Non_Null": df_raw.notna().sum().values,
    "Missing": df_raw.isna().sum().values,
})

audit_types["Missing_%"] = (
    100 * audit_types["Missing"] / len(df_raw)
).round(2)

display(audit_types)

In [ ]:
print("=" * 70)
print("PATIENT STRUCTURE")
print("=" * 70)

print("Unique Code values:")
print(df_raw["Code"].nunique(dropna=True))

print("\nUnique Study values:")
print(df_raw["Study"].nunique(dropna=True))

display(
    df_raw.groupby("Study", dropna=False)
          .agg(
              Rows=("Code", "size"),
              Unique_Codes=("Code", "nunique")
          )
          .reset_index()
)

In [ ]:
df = df_raw.copy(deep=True)

df["Patient_ID"] = (
    df["Study"].astype(str).str.strip()
    + "__"
    + df["Code"].astype(str).str.strip()
)

print("Rows:", len(df))
print("Unique Patient_ID:", df["Patient_ID"].nunique())

display(df[["Study", "Code", "Patient_ID"]].head())

In [ ]:
duplicate_patient_day = (
    df.duplicated(
        subset=["Patient_ID", "DOI"],
        keep=False
    )
)

n_dup_rows = duplicate_patient_day.sum()

print("Rows involved in duplicate Patient_ID × DOI combinations:")
print(n_dup_rows)

if n_dup_rows > 0:
    display(
        df.loc[
            duplicate_patient_day,
            ["Patient_ID", "DOI", "PLT", "vir",
             "Severe_dengue", "Leakage"]
        ]
        .sort_values(["Patient_ID", "DOI"])
        .head(50)
    )
else:
    print("No duplicate patient-day records detected.")

In [ ]:
doi_numeric = pd.to_numeric(df["DOI"], errors="coerce")

print("DOI minimum:", doi_numeric.min())
print("DOI maximum:", doi_numeric.max())

doi_distribution = (
    doi_numeric.value_counts(dropna=False)
               .sort_index()
               .rename_axis("DOI")
               .reset_index(name="Observations")
)

display(doi_distribution)

In [ ]:
patient_obs = (
    df.groupby("Patient_ID")
      .agg(
          N_Observations=("DOI", "size"),
          First_DOI=("DOI", "min"),
          Last_DOI=("DOI", "max"),
          Platelet_Observations=("PLT", lambda x: x.notna().sum()),
          Viremia_Observations=("vir", lambda x: x.notna().sum()),
      )
      .reset_index()
)

display(patient_obs.describe())

print("\nPatients with only one observation:")
print((patient_obs["N_Observations"] == 1).sum())

print("\nPatients with >= 2 observations:")
print((patient_obs["N_Observations"] >= 2).sum())

print("\nPatients with >= 3 observations:")
print((patient_obs["N_Observations"] >= 3).sum())

In [ ]:
missingness = pd.DataFrame({
    "Variable": df.columns,
    "N_Missing": df.isna().sum().values,
    "Percent_Missing": (
        df.isna().mean().values * 100
    )
})

missingness["Percent_Missing"] = (
    missingness["Percent_Missing"].round(2)
)

missingness = missingness.sort_values(
    "Percent_Missing",
    ascending=False
)

display(missingness)

In [ ]:
categorical_vars = [
    "Study",
    "Sex",
    "Serotype",
    "Serology",
    "Severe_dengue",
    "Leakage"
]

for col in categorical_vars:
    print("\n" + "=" * 70)
    print(col)
    print("=" * 70)

    print(
        df[col]
        .value_counts(dropna=False)
        .head(30)
    )

In [ ]:
numeric_candidates = [
    "Age",
    "Enrol_DOI",
    "DOI",
    "PLT",
    "vir",
    "doi_severe",
    "doi_leakage",
]

numeric_audit = []

for col in numeric_candidates:
    converted = pd.to_numeric(df[col], errors="coerce")

    numeric_audit.append({
        "Variable": col,
        "Original_non_missing": df[col].notna().sum(),
        "Numeric_non_missing": converted.notna().sum(),
        "Min": converted.min(),
        "Median": converted.median(),
        "Max": converted.max(),
    })

numeric_audit = pd.DataFrame(numeric_audit)

display(numeric_audit)

In [ ]:
age_num = pd.to_numeric(df["Age"], errors="coerce")
doi_num = pd.to_numeric(df["DOI"], errors="coerce")
plt_num = pd.to_numeric(df["PLT"], errors="coerce")
vir_num = pd.to_numeric(df["vir"], errors="coerce")

flags = {
    "Age_negative": (age_num < 0).sum(),
    "DOI_negative": (doi_num < 0).sum(),
    "Platelet_negative": (plt_num < 0).sum(),
    "Viremia_negative": (vir_num < 0).sum(),
}

display(pd.DataFrame(
    flags.items(),
    columns=["Integrity_Check", "N_Flagged"]
))

In [ ]:
patient_outcomes = (
    df.groupby("Patient_ID")
      .agg({
          "Severe_dengue": lambda x: x.dropna().unique().tolist(),
          "Leakage": lambda x: x.dropna().unique().tolist(),
          "Serotype": lambda x: x.dropna().unique().tolist(),
          "Age": "first",
          "Sex": "first",
          "Study": "first",
      })
      .reset_index()
)

print("Patient-level rows:", len(patient_outcomes))

display(patient_outcomes.head())

In [ ]:
coverage = (
    df.assign(
        DOI_numeric=pd.to_numeric(df["DOI"], errors="coerce")
    )
    .dropna(subset=["DOI_numeric"])
    .groupby("DOI_numeric")
    .agg(
        Patients=("Patient_ID", "nunique"),
        Platelet_available=("PLT", lambda x: x.notna().sum()),
        Viremia_available=("vir", lambda x: x.notna().sum())
    )
    .reset_index()
    .sort_values("DOI_numeric")
)

display(coverage)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    coverage["DOI_numeric"],
    coverage["Patients"],
    marker="o",
    label="Patients"
)

ax.plot(
    coverage["DOI_numeric"],
    coverage["Platelet_available"],
    marker="o",
    label="Platelet observations"
)

ax.plot(
    coverage["DOI_numeric"],
    coverage["Viremia_available"],
    marker="o",
    label="Viremia observations"
)

ax.set_xlabel("Day of illness (DOI)")
ax.set_ylabel("Number of observations")
ax.set_title("Longitudinal measurement coverage")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
audit_summary = {
    "Dataset": DATA_FILENAME,
    "Rows": len(df_raw),
    "Columns": df_raw.shape[1],
    "Unique_patients": df["Patient_ID"].nunique(),
    "Studies": df["Study"].nunique(dropna=True),
    "Duplicate_patient_day_rows": int(n_dup_rows),
    "Random_seed": RANDOM_SEED,
}

print("=" * 70)
print("NOTEBOOK 01 AUDIT SUMMARY")
print("=" * 70)

for key, value in audit_summary.items():
    print(f"{key:30s}: {value}")